In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from nltk import bigrams
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
df = pd.read_csv("fake_news_clean.csv")

df["contenido"] = df["contenido"].astype(str)
df["titulo"] = df["titulo"].astype(str)

In [ ]:
def count_tokens(text):
    return len(re.findall(r"\w+", text))

df["token_count"] = df["contenido"].apply(count_tokens)

plt.figure(figsize=(7,4))
sns.histplot(df["token_count"], bins=50)
plt.title("Distribución de tokens por noticia")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(data=df, x="token_count", hue="label", bins=50, kde=True)
plt.title("Tokens por clase (FAKE vs REAL)")
plt.show()

In [ ]:
df["char_length"] = df["contenido"].str.len()

plt.figure(figsize=(7,4))
sns.histplot(df["char_length"], bins=50)
plt.title("Distribución de longitud en caracteres")
plt.show()

In [ ]:
def lexical_richness(text):
    tokens = re.findall(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]+", text.lower())
    if len(tokens) == 0:
        return 0
    return len(set(tokens)) / len(tokens)

df["ttr"] = df["contenido"].apply(lexical_richness)

plt.figure(figsize=(7,4))
sns.histplot(df["ttr"], bins=50)
plt.title("Distribución de riqueza léxica (TTR)")
plt.show()

In [ ]:
def clean_words(text):
    return re.findall(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]+", text.lower())

all_words = []
for t in df["contenido"]:
    all_words.extend(clean_words(t))

word_freq = Counter(all_words).most_common(20)
print("Top 20 palabras más frecuentes:")
for w,f in word_freq:
    print(w,":",f)

In [ ]:

fake_words = Counter([w for t in df[df["label"]==0]["contenido"] for w in clean_words(t)]).most_common(20)
real_words = Counter([w for t in df[df["label"]==1]["contenido"] for w in clean_words(t)]).most_common(20)

print("\nTop 20 palabras FAKE:")
print(fake_words)
print("\nTop 20 palabras REAL:")
print(real_words)

In [ ]:
def get_bigrams(text):
    words = clean_words(text)
    return list(bigrams(words))

all_bigrams = []
for t in df["contenido"].sample(2000):
    all_bigrams.extend(get_bigrams(t))

bigram_freq = Counter(all_bigrams).most_common(20)
print("\nTop 20 bigramas más frecuentes:")
for b,f in bigram_freq:
    print(b,":",f)

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x="label", y="token_count")
plt.title("Distribución de longitud por clase (tokens)")
plt.show()

In [ ]:
Q1 = df["token_count"].quantile(0.25)
Q3 = df["token_count"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

outliers = df[(df["token_count"] < lower_bound) | (df["token_count"] > upper_bound)]
print("\nCantidad de outliers:", len(outliers))

In [ ]:
over_bert_limit = df[df["token_count"] > 512]
print("\nCantidad de textos que exceden 512 tokens:", len(over_bert_limit))

plt.figure(figsize=(7,4))
sns.histplot(df["token_count"], bins=80)
plt.axvline(512, color="red")
plt.title("Longitud vs límite de BERT (512 tokens)")
plt.show()

In [ ]:
MAX_TOKENS = 2000

def truncate_text(text, max_len=MAX_TOKENS):
    words = text.split()
    if len(words) > max_len:
        return " ".join(words[:max_len])
    return text

df["contenido"] = df["contenido"].astype(str).apply(truncate_text)

# recalcular tokens para análisis posterior
df["token_count"] = df["contenido"].apply(lambda x: len(x.split()))



In [ ]:
df["label"].value_counts()

In [ ]:
import pandas as pd

# Cargar el dataset limpio
df = pd.read_csv("fake_news_clean.csv")

# Guardar con el nuevo nombre
df.to_csv("fake_news_final.csv", index=False)

print("Archivo guardado como fake_news_final.csv")
